In [ ]:
import pandas as pd

df_sisam_2022 = pd.read_csv("sisam_municipio_reanalise_diario_2022-01-01_2022-12-31.csv")
df_sisam_2023 = pd.read_csv("sisam_municipio_reanalise_diario_2023-01-01_2023-12-31.csv")
df_sisam_2024 = pd.read_csv("sisam_municipio_reanalise_diario_2024-01-01_2024-12-31.csv")

df_sisam = pd.concat([df_sisam_2022, df_sisam_2023, df_sisam_2024], ignore_index=True)

print("Shape combinado:", df_sisam.shape)
print("\nColumnas:", df_sisam.columns.tolist())
df_sisam.head(10)

Shape combinado: (885934, 10)

Columnas: ['CD_MUN', 'municipio', 'estado', 'data', 'pm10_reanalise', 'pm2_5_reanalise', 'o3_reanalise', 'no2_reanalise', 'co_reanalise', 'so2_reanalise']


,CD_MUN,municipio,estado,data,pm10_reanalise,pm2_5_reanalise,o3_reanalise,no2_reanalise,co_reanalise,so2_reanalise
0,1200013,ACRELÂNDIA,ACRE,31/12/2022,15.93,12.08,14.30,0.47,0.12,0.05
1,1200054,ASSIS BRASIL,ACRE,31/12/2022,14.66,10.93,10.81,0.15,0.12,0.02
2,1200104,BRASILÉIA,ACRE,31/12/2022,13.91,10.42,11.21,0.24,0.12,0.04
3,1200138,BUJARI,ACRE,31/12/2022,16.26,12.31,12.78,0.41,0.12,0.06
4,1200179,CAPIXABA,ACRE,31/12/2022,13.97,10.54,13.26,0.31,0.12,0.05
5,1200203,CRUZEIRO DO SUL,ACRE,31/12/2022,26.87,21.09,8.38,0.23,0.12,0.03
6,1200252,EPITACIOLÂNDIA,ACRE,31/12/2022,13.88,10.40,11.18,0.24,0.12,0.04
7,1200302,FEIJÓ,ACRE,31/12/2022,22.47,17.35,9.74,0.30,0.12,0.01
8,1200328,JORDÃO,ACRE,31/12/2022,20.05,15.32,7.72,0.08,0.12,0.00
9,1200336,MÂNCIO LIMA,ACRE,31/12/2022,27.87,21.89,8.52,0.23,0.12,0.03


In [ ]:
print("Valores nulos por columna:")
print(df_sisam.isnull().sum())

print("\nEstados presentes:", df_sisam["estado"].unique())
print("Municipios únicos:", df_sisam["municipio"].nunique())

# la fecha viene en formato dd/mm/aaaa (día primero) -- distinto al de BDQueimadas
df_sisam["data"] = pd.to_datetime(df_sisam["data"], format="%d/%m/%Y", errors="coerce")
print("\nFechas inválidas tras conversión:", df_sisam["data"].isna().sum())
print("Rango de fechas:", df_sisam["data"].min(), "→", df_sisam["data"].max())

print("\nEstadísticas de las variables químicas:")
print(df_sisam[["pm10_reanalise", "pm2_5_reanalise", "o3_reanalise",
                 "no2_reanalise", "co_reanalise", "so2_reanalise"]].describe())

Valores nulos por columna:
CD_MUN             0
municipio          0
estado             0
data               0
pm10_reanalise     0
pm2_5_reanalise    0
o3_reanalise       0
no2_reanalise      0
co_reanalise       0
so2_reanalise      0
dtype: int64

Estados presentes: ['ACRE' 'AMAPÁ' 'AMAZONAS' 'MARANHÃO' 'MATO GROSSO' 'PARÁ' 'RONDÔNIA'
 'RORAIMA' 'TOCANTINS']
Municipios únicos: 805

Fechas inválidas tras conversión: 0
Rango de fechas: 2022-01-01 00:00:00 → 2024-12-31 00:00:00

Estadísticas de las variables químicas:
       pm10_reanalise  pm2_5_reanalise   o3_reanalise  no2_reanalise  \
count   885934.000000    885934.000000  885934.000000  885934.000000   
mean        23.871912        17.352832      30.853878       1.394829   
std         35.847633        26.038334      14.754061       2.285618   
min          0.590000         0.410000       2.340000       0.010000   
25%         11.480000         8.130000      19.400000       0.390000   
50%         16.160000        11.660000      

In [ ]:
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')

df_sisam["municipio_norm"] = df_sisam["municipio"].apply(normalizar_texto)

print("Municipios únicos en SISAM:", df_sisam["municipio_norm"].nunique())
print(df_sisam["municipio_norm"].sort_values().unique()[:15])

Municipios únicos en SISAM: 805
['ABAETETUBA' 'ABEL FIGUEIREDO' 'ABREULANDIA' 'ACAILANDIA' 'ACARA'
 'ACORIZAL' 'ACRELANDIA' 'AFONSO CUNHA' 'AFUA' 'AGUA AZUL DO NORTE'
 'AGUA BOA' 'AGUA DOCE DO MARANHAO' 'AGUIARNOPOLIS' 'ALCANTARA'
 'ALDEIAS ALTAS']


In [ ]:
df_deter= pd.read_csv("deter_limpio_merge.csv")

In [ ]:
# verificación cruzada con DETER (mismo patrón que usamos con BDQueimadas)
municipios_deter = set(df_deter["municipio_norm"].unique())
municipios_sisam = set(df_sisam["municipio_norm"].unique())

print(f"Municipios en DETER: {len(municipios_deter)}")
print(f"Municipios en SISAM: {len(municipios_sisam)}")
print(f"Municipios en común: {len(municipios_deter & municipios_sisam)}")
print(f"En DETER pero NO en SISAM: {len(municipios_deter - municipios_sisam)}")
print(sorted(municipios_deter - municipios_sisam))

Municipios en DETER: 424
Municipios en SISAM: 805
Municipios en común: 424
En DETER pero NO en SISAM: 0
[]


In [ ]:
columnas_finales = [
    "data", "municipio_norm", "estado",
    "pm10_reanalise", "pm2_5_reanalise", "o3_reanalise",
    "no2_reanalise", "co_reanalise", "so2_reanalise"
]

df_sisam_limpio = df_sisam[columnas_finales].copy()

df_sisam_limpio.to_csv("sisam_limpio.csv", index=False, encoding="utf-8")
print("sisam_limpio.csv guardado:", df_sisam_limpio.shape)
print(df_sisam_limpio.dtypes)

sisam_limpio.csv guardado: (885934, 9)
data               datetime64[ns]
municipio_norm             object
estado                     object
pm10_reanalise            float64
pm2_5_reanalise           float64
o3_reanalise              float64
no2_reanalise             float64
co_reanalise              float64
so2_reanalise             float64
dtype: object
